In [63]:
from langchain_aws import ChatBedrockConverse
from langchain_ollama import ChatOllama
from typing import TypedDict, Literal,List,Dict,Annotated
from langgraph.graph import StateGraph,START,END,MessagesState
from pydantic import BaseModel,Field
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
import sqlite3
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage,BaseMessage,ToolMessage
import operator
from langgraph.prebuilt.tool_node import ToolNode
from langchain_core.tools import tool
import uuid
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command,interrupt



In [64]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=100)

In [65]:
class StateSchema(TypedDict):
    restaurant: str
    distance_km: float
    delivery_fee: float
    summary: str
    
  
    

In [66]:
def order_process(state:StateSchema):
    delivery_fee = state['distance_km'] * 100 
    return {'delivery_fee': delivery_fee}

def generate_summary(state:StateSchema):
    prompt = f'generate a very short summary for order details - {state} '
    summary = llm.invoke(prompt)
    return {'summary':summary}


In [67]:
graph = StateGraph(StateSchema)
graph.add_node('order_process_node',order_process)
graph.add_node('generate_summary_node',generate_summary)

graph.add_edge(START,'order_process_node')
graph.add_edge('order_process_node','generate_summary_node')
graph.add_edge('generate_summary_node',END)

conn = sqlite3.connect("database/data.db",check_same_thread=False)
workflow = graph.compile(checkpointer=SqliteSaver(conn=conn))

In [68]:
print(workflow.get_graph().draw_ascii())

      +-----------+        
      | __start__ |        
      +-----------+        
            *              
            *              
            *              
  +--------------------+   
  | order_process_node |   
  +--------------------+   
            *              
            *              
            *              
+-----------------------+  
| generate_summary_node |  
+-----------------------+  
            *              
            *              
            *              
       +---------+         
       | __end__ |         
       +---------+         


In [69]:
thread_id = uuid.uuid4()
config = {'configurable':{'thread_id':thread_id}}
res = workflow.invoke({'restaurant':'haldirams restaurant','distance_km':3},config=config)
res

{'restaurant': 'haldirams restaurant',
 'distance_km': 3,
 'delivery_fee': 300,
 'summary': AIMessage(content="Order to Haldiram's Restaurant at 3 km distance with a delivery fee of 300.", additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-09-20T09:55:50.766123357Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1360640821, 'load_duration': 190897227, 'prompt_eval_count': 64, 'prompt_eval_duration': 361340000, 'eval_count': 23, 'eval_duration': 806625000, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a0be3e-4e1c-7ee0-aff7-fa4661149390-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 23, 'total_tokens': 87})}

In [70]:
chat_history = list(workflow.get_state_history(config=config))
chat_history.reverse()
for msg in chat_history:
    print(msg)

StateSnapshot(values={}, next=('__start__',), config={'configurable': {'thread_id': 'b03b1151-3695-4755-a9ec-b7304ff0b83a', 'checkpoint_ns': '', 'checkpoint_id': '1f1b4d97-54c0-6052-bfff-857656ea8882'}}, metadata={'source': 'input', 'step': -1, 'parents': {}}, created_at='2026-09-20T09:55:49.401388+00:00', parent_config=None, tasks=(PregelTask(id='2111f77a-3149-baa1-b6ef-4df8c65ca2a8', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'restaurant': 'haldirams restaurant', 'distance_km': 3}),), interrupts=())
StateSnapshot(values={'restaurant': 'haldirams restaurant', 'distance_km': 3}, next=('order_process_node',), config={'configurable': {'thread_id': 'b03b1151-3695-4755-a9ec-b7304ff0b83a', 'checkpoint_ns': '', 'checkpoint_id': '1f1b4d97-54c2-640c-8000-53c42712c5d1'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-09-20T09:55:49.402300+00:00', parent_config={'configurable': {'thread_id': 'b03b1151-3695-4755-

In [71]:
chat_history = list(workflow.get_state_history(config=config))
chat_history.reverse()
config_fork = chat_history[2].config
new_config = workflow.update_state(config=config_fork,values={'delivery_fee':10})
res = workflow.invoke(None,config=new_config)
res

{'restaurant': 'haldirams restaurant',
 'distance_km': 3,
 'delivery_fee': 10,
 'summary': AIMessage(content="Order to Haldiram's Restaurant at 3 km distance with a delivery fee of $10.", additional_kwargs={}, response_metadata={'model': 'qwen2.5:1.5b', 'created_at': '2026-09-20T09:55:51.979609406Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1176614502, 'load_duration': 292900489, 'prompt_eval_count': 63, 'prompt_eval_duration': 112555000, 'eval_count': 22, 'eval_duration': 768717000, 'logprobs': None, 'model_name': 'qwen2.5:1.5b', 'model_provider': 'ollama'}, id='lc_run--01a0be3e-5391-70b3-ac7f-14f977926c22-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 22, 'total_tokens': 85})}